# Universal Grammar Generative Model

In [1]:
INSTALL = false

if INSTALL
    using Pkg
    Pkg.activate("psyc261")
    Pkg.add(["JLD2", "Distributions", "ProgressMeter", "Parameters", 
            "Random", "Gen", "Plots", "PyCall", "Conda", "JSON3"])
end

In [2]:
using Pkg
Pkg.activate("psyc261")
using JLD2
using Distributions
using ProgressMeter
using Parameters
using Random
using Gen, Plots
using PyCall
np = pyimport("numpy")

  Activating project at `~/aom/psyc261`


PyObject <module 'numpy' from '/opt/conda/lib/python3.11/site-packages/numpy/__init__.py'>

## Generative Model

In [3]:
const NOUNS = ["dog", "cat", "fox"]
const VERBS_TRANS = ["see", "chase", "attack"]
const VERBS_INTR = ["sleep", "run"]
const ADJECTIVES = ["big", "small"]
const WH_WORD = "what"

"what"

In [4]:
@gen function corpus_model(num_sentences::Int)
    # === Universal Grammar paramaters (theta) ===
    # Head direction: 1 = Verb-Object (VO), 0 = Object-Verb(OV)
    hd = @trace(bernoulli(0.5), :head_direction)

    # Subject drop: 1 = subject may be dropped, 0 = subject always present
    sd = @trace(bernoulli(0.5), :subject_drop)

    # Adjective order: 1 = Adjective-Noun (AN), 0 = Noun-Adjective (NA)
    ao = @trace(bernoulli(0.5), :adj_order)

    # Wh-fronting in question: 1 = wh-word at front, 0 = in-situ
    wh = @trace(bernoulli(0.5), :wh_fronting)
    
    sentences = Vector{Vector{String}}(undef, num_sentences)

    for i in 1:num_sentences
        # Utterance type: 1 = transitive declarative, 2 = intransitive, 3 = transitive question
        utt_type = @trace(categorical([0.4, 0.3, 0.3]), (:utt_type, i))

        # Sample lexical items
        subj_idx = @trace(categorical(fill(1/length(NOUNS), length(NOUNS))), (:subj, i))
        obj_idx = @trace(categorical(fill(1/length(NOUNS), length(NOUNS))), (:obj, i))
        adj_idx = @trace(categorical(fill(1/length(ADJECTIVES), length(ADJECTIVES))), (:adj, i))

        if utt_type == 1 || utt_type == 3
            v_idx = @trace(categorical(fill(1/length(VERBS_TRANS), length(VERBS_TRANS))), (:v_trans, i))
            verb = VERBS_TRANS[v_idx]
        else
            v_idx = @trace(categorical(fill(1/length(VERBS_INTR), length(VERBS_INTR))), (:v_intr, i))
            verb = VERBS_INTR[v_idx]
        end

        subj = NOUNS[subj_idx]
        obj = NOUNS[obj_idx]
        adj = ADJECTIVES[adj_idx]

        # Build NP for subject and object, respecting adjective order
        function make_np(noun::String, adj::String)
            if ao == 1
                return [adj, noun] # Adj N
            else
                return [noun, adj] # N Adj
            end
        end

        subj_np = make_np(subj, adj)
        obj_np = make_np(obj, adj)

        # Drop subject if SD = 1
        drop_subj = (sd == 1) ? @trace(bernoulli(0.5), (:drop_subj, i)) : 0

        # Base clause word order for transitive
        if utt_type == 1 || utt_type == 3
            # Verb-Object vs. Object-Verb in the VP
            vp = (hd == 1) ? vcat([verb], obj_np) : vcat(obj_np, [verb])
        else
            # Intransitive: just the verb
            vp = [verb]
        end

        clause = Vector{String}()

        if drop_subj == 0
            append!(clause, subj_np)
        end
        append!(clause, vp)

        # Handle questions
        if utt_type == 3
            if wh == 1
                # wh-fronting
                clause = vcat([WH_WORD], clause)
            else
                # wh in-situ (append at the end)
                push!(clause, WH_WORD)
            end
        end

        sentences[i] = clause
    end

    return sentences
end

DynamicDSLFunction{Any}(Dict{Symbol, Any}(), Dict{Symbol, Any}(), Type[Int64], false, Union{Nothing, Some{Any}}[nothing], var"##corpus_model#292", Bool[0], false)

In [5]:
trace = simulate(corpus_model, (10,))
sentences = get_retval(trace)
theta = (
    hd = trace[:head_direction],
    sd = trace[:subject_drop],
    ao = trace[:adj_order],
    wh = trace[:wh_fronting]
)

(hd = false, sd = true, ao = false, wh = false)

In [12]:
@gen function corpus_given_theta(hd::Bool, sd::Bool, ao::Bool, wh::Bool, num_sentences::Int)
    sentences = Vector{Vector{String}}(undef, num_sentences)

    for i in 1:num_sentences
        utt_type = @trace(categorical([0.4, 0.3, 0.3]), (:utt_type, i))

        subj_idx = @trace(categorical(fill(1/length(NOUNS), length(NOUNS))), (:subj, i))
        obj_idx = @trace(categorical(fill(1/length(NOUNS), length(NOUNS))), (:obj, i))
        adj_idx = @trace(categorical(fill(1/length(ADJECTIVES), length(ADJECTIVES))), (:adj, i))

        if utt_type == 1 || utt_type == 3
            v_idx = @trace(categorical(fill(1/length(VERBS_TRANS), length(VERBS_TRANS))), (:v_trans, i))
            verb = VERBS_TRANS[v_idx]
        else
            v_idx = @trace(categorical(fill(1/length(VERBS_INTR), length(VERBS_INTR))), (:v_intr, i))
            verb = VERBS_INTR[v_idx]
        end

        subj = NOUNS[subj_idx]
        obj = NOUNS[obj_idx]
        adj = ADJECTIVES[adj_idx]

        function make_np(noun::String, adj::String)
            if ao == 1
                return [adj, noun] # Adj N
            else
                return [noun, adj] # N Adj
            end
        end
        
        subj_np = make_np(subj, adj)
        obj_np = make_np(obj, adj)

        drop_subj = (sd == 1) ? @trace(bernoulli(0.5), (:drop_subj, i)) : 0

        if utt_type == 1 || utt_type == 3
            vp = (hd == 1) ? vcat([verb], obj_np) : vcat(obj_np, [verb])
        else
            vp = [verb]
        end

        clause = Vector{String}()

        if drop_subj == 0
            append!(clause, subj_np)
        end
        append!(clause, vp)

        if utt_type == 3
            if wh == 1
                clause = vcat([WH_WORD], clause)
            else
                push!(clause, WH_WORD)
            end
        end

        sentences[i] = clause
    end

    return sentences
end

DynamicDSLFunction{Any}(Dict{Symbol, Any}(), Dict{Symbol, Any}(), Type[Bool, Bool, Bool, Bool, Int64], false, Union{Nothing, Some{Any}}[nothing, nothing, nothing, nothing, nothing], var"##corpus_given_theta#294", Bool[0, 0, 0, 0, 0], false)

In [13]:
using JSON3

SAVE = false

if SAVE
    theta = (hd=false, sd=false, ao=true, wh=true)
    num_sentences = 100
    
    trace = simulate(corpus_given_theta, (theta.hd, theta.sd, theta.ao, theta.wh, num_sentences))
    corpus = get_retval(trace)
    
    open("data/ug_language1.json", "w") do io
        JSON3.write(io, Dict("theta" => theta, "sentences" => corpus))
    end
end

3565